In [ ]:
# 1) mount drive
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/GutBrainIE/Train/NER

# PyTorch с CUDA
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# GLiNER stack
!pip install \
  gliner==0.1.12 \
  "transformers>=4.38.2,<=4.45.2" \
  "huggingface_hub>=0.21.4" \
  onnxruntime-gpu \
  sentencepiece \
  tqdm

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Mounted at /content/drive
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_508/1178823231.py", line 5, in <cell line: 0>
    get_ipython().run_line_magic('cd', '/content/drive/MyDrive/GutBrainIE/Train/NER')
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2418, in run_line_magic
    result = fn(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^
  File "<decorator-gen-85>", line 2, in cd
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magic.py", line 187, in <lambda>
    call = lambda f, *a, **k: f(*a, **k)
                              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/magics/osm.py", line 342, in cd
    oldcwd = os.getcwd()
             ^^^^^^^^^^^
OSError: [Errno 107] Transport endpoint is not connected

During handling of the abo

In [ ]:
import json
from gliner import GLiNER

import torch
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
import os

from importlib.metadata import version
version('GLiNER')

# Set the GLiNER model to be used (from HuggingFace)
model = GLiNER.from_pretrained("numind/NuNerZero")
model_name = "NuNerZero"

# Define the confidence threshold to be used in evaluation
THRESHOLD = 0.65

# Define whether the code should be used for fine-tuning
finetune_model = False

# Define the path to articles for which the final trained will generate predicted entities
generate_predictions = True
PATH_ARTICLES = "../../Articles/json_format/articles_test.json"
PATH_OUTPUT_NER_PREDICTIONS = "../../Predictions/NER/predicted_entities_test_0.65.json"

print('## LOADING TRAINING DATA ##')
PATH_GOLD_TRAIN = "data/train_gold.json"
PATH_SILVER_TRAIN = "data/train_silver.json"
PATH_SILVER_TRAIN_2025 = "data/train_silver_2025.json"
PATH_BRONZE_TRAIN = "data/train_bronze.json"
PATH_DEV = "data/dev.json"

with open(PATH_GOLD_TRAIN, 'r', encoding='utf-8') as file:
	train_gold = json.load(file)

with open(PATH_SILVER_TRAIN, 'r', encoding='utf-8') as file:
	train_silver = json.load(file)

with open(PATH_SILVER_TRAIN_2025, 'r', encoding='utf-8') as file:
	train_silver_2025 = json.load(file)

with open(PATH_BRONZE_TRAIN, 'r', encoding='utf-8') as file:
	train_bronze = json.load(file)

with open(PATH_DEV, 'r', encoding='utf-8') as file:
	eval_data = json.load(file)

# Set the data to be used for training
# Here we used the platinum, gold, and silver sets
train_data = train_gold + train_silver + train_silver_2025

# converting entities-level train data to token-level
new_data = []
for d in train_data:
    new_ner = []
    for s, f, c in d["ner"]:
        for i in range(s, f + 1):
            # labels are intended to be lower-case
            new_ner.append((i, i, c.lower()))
    new_d = {
        "tokenized_text": d["tokenized_text"],
        "ner": new_ner,
    }
    new_data.append(new_d)
train_data = new_data

# converting entities-level eval data to token-level
new_data = []
for d in eval_data:
    new_ner = []
    for s, f, c in d["ner"]:
        for i in range(s, f + 1):
            # labels are intended to be lower-case
            new_ner.append((i, i, c.lower()))
    new_d = {
        "tokenized_text": d["tokenized_text"],
        "ner": new_ner,
    }
    new_data.append(new_d)
eval_data = new_data

from types import SimpleNamespace

# Define the hyperparameters in a config variable
config = SimpleNamespace(
    num_steps=3000, # regulate number train, eval steps depending on the data size
    eval_every=500,

    train_batch_size=8, # regulate batch size depending on GPU memory available.

    max_len=768, # maximum sentence length. 2048 for NuNerZero_long_context

    save_directory="logs", # log dir
    device='cuda' if torch.cuda.is_available() else 'cpu', #'cuda', # training device - cpu or cuda

    warmup_ratio=0.1, # Other parameters
    lr_encoder=1e-5,
    lr_others=5e-5,
    freeze_token_rep=False,

    max_types=15,
    shuffle_types=True,
    random_drop=True,
    max_neg_type_ratio=1,
)


# modify this to your own test data!
# don't forget to do the same preprocessing as for the train data:
# * converting entities-level data to token-level data
# * making entity_types lower-cased!!!
eval_data = {
    "entity_types": [
        "anatomical location",
        "animal",
        "biomedical technique",
        "bacteria",
        "chemical",
        "dietary supplement",
        "ddf",
        "drug",
        "food",
        "gene",
        "human",
        "microbiome",
        "statistical technique",
    ],
    "samples": eval_data
}

print('## DEFINING TRAINING FUNCTION ##')
def train(model, config, train_data, eval_data=None):
    model = model.to(config.device)

    # Set sampling parameters from config
    model.set_sampling_params(
        max_types=config.max_types,
        shuffle_types=config.shuffle_types,
        random_drop=config.random_drop,
        max_neg_type_ratio=config.max_neg_type_ratio,
        max_len=config.max_len
    )

    model.train()

    # Initialize data loaders
    train_loader = model.create_dataloader(train_data, batch_size=config.train_batch_size, shuffle=True)

    # Optimizer
    optimizer = model.get_optimizer(config.lr_encoder, config.lr_others, config.freeze_token_rep)

    pbar = tqdm(range(config.num_steps))

    if config.warmup_ratio < 1:
        num_warmup_steps = int(config.num_steps * config.warmup_ratio)
    else:
        num_warmup_steps = int(config.warmup_ratio)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=config.num_steps
    )

    iter_train_loader = iter(train_loader)

    for step in pbar:
        try:
            x = next(iter_train_loader)
        except StopIteration:
            iter_train_loader = iter(train_loader)
            x = next(iter_train_loader)

        for k, v in x.items():
            if isinstance(v, torch.Tensor):
                x[k] = v.to(config.device)

        loss = model(x)  # Forward pass

        # Check if loss is nan
        if torch.isnan(loss):
            continue

        loss.backward()  # Compute gradients
        optimizer.step()  # Update parameters
        scheduler.step()  # Update learning rate schedule
        optimizer.zero_grad()  # Reset gradients

        description = f"step: {step} | epoch: {step // len(train_loader)} | loss: {loss.item():.2f}"
        pbar.set_description(description)

        if (step + 1) % config.eval_every == 0:

            model.eval()

            if eval_data is not None:
                results, f1 = model.evaluate(eval_data["samples"], flat_ner=True, threshold=THRESHOLD, batch_size=32,
                                     entity_types=eval_data["entity_types"])

                print(f"Step={step}\n{results}")

            if not os.path.exists(config.save_directory):
                os.makedirs(config.save_directory)

            model.save_pretrained(f"{config.save_directory}/finetuned_{step}")

            model.train()

if finetune_model:
    print('## LAUNCHING TRAINING ##')
    train(model, config, train_data, eval_data)

    print('## SAVING TRAINED MODEL ##')
    output_path = f"outputs/{model_name}_finetuned"
    model.save_pretrained(output_path)
    os.system(f'cp {output_path}/gliner_config.json {output_path}/config.json')
    md = GLiNER.from_pretrained(output_path, local_files_only=True)

if generate_predictions:
    output_path = f"outputs/{model_name}_finetuned"
    print(f"## LOADING PRE-TRAINED MODEL {output_path} ##")
    md = GLiNER.from_pretrained(output_path, local_files_only=True)

    print(f"## GENERATING NER PREDICTIONS FOR {PATH_ARTICLES}")
    with open(PATH_ARTICLES, 'r', encoding='utf-8') as file:
        articles = json.load(file)

    print(f"len(articles): {len(articles)}")
    entity_labels = eval_data['entity_types']

    # Dictionary to hold predicted entities
    # PMID -> {{'start_idx': ..., 'end_idx': ..., 'text_span': ..., 'entity_label': ..., 'score': ...}, ...}
    predictions = {}

    for pmid, content in tqdm(articles.items(), total=len(articles), desc="Predicting entities..."):
        title = content['title']
        abstract = content['abstract']

        # Predict entities
        title_entities = md.predict_entities(title, entity_labels, threshold=THRESHOLD, flat_ner=True, multi_label=False)
        abstract_entities = md.predict_entities(abstract, entity_labels, threshold=THRESHOLD, flat_ner=True, multi_label=False)

        # Adjust indices for predicted entities in the abstract
        for entity in abstract_entities:
            entity['start'] += len(title) + 1
            entity['end'] += len(title) + 1

        # Remove duplicates from predicted entities
        unique_entities = []
        seen_entities = set()

        # Remove duplicates from title entities and add tag field with value 't'
        for entity in title_entities:
            key = (entity['start'], entity['end'], entity['text'], entity['label'], entity['score'])
            if key not in seen_entities:
                tmp_entity = {
                    'start_idx': entity['start'],
                    'end_idx': entity['end'],
                    'tag': 't',
                    'text_span': entity['text'],
                    'entity_label': entity['label'],
                    'score': entity['score']
                }
                unique_entities.append(tmp_entity)
                seen_entities.add(key)

        # Remove duplicates from abstract entities and add tag field with value 'a'
        for entity in abstract_entities:
            key = (entity['start'], entity['end'], entity['text'], entity['label'], entity['score'])
            if key not in seen_entities:
                tmp_entity = {
                    'start_idx': entity['start'],
                    'end_idx': entity['end'],
                    'tag': 'a',
                    'text_span': entity['text'],
                    'entity_label': entity['label'],
                    'score': entity['score']
                }
                unique_entities.append(tmp_entity)
                seen_entities.add(key)

        predictions[pmid] = unique_entities
        articles[pmid]['pred_entities'] = unique_entities

    # Convert any non-serializable data if necessary
    def default_serializer(obj):
        if isinstance(obj, set):
            return list(obj)
        # Add other types if needed
        raise TypeError(f'Type {type(obj)} not serializable')

    with open(PATH_OUTPUT_NER_PREDICTIONS, 'w', encoding='utf-8') as f:
        json.dump(articles, f, ensure_ascii=False, indent=2, default=default_serializer)

    print(f"## Predictions have been exported in JSON format to '/{PATH_OUTPUT_NER_PREDICTIONS}' ##")



## LOADING TRAINING DATA ##
## DEFINING TRAINING FUNCTION ##
## LOADING PRE-TRAINED MODEL outputs/NuNerZero_finetuned ##
## GENERATING NER PREDICTIONS FOR ../../Articles/json_format/articles_test.json
len(articles): 80


Predicting entities...:  16%|█▋        | 13/80 [01:16<06:52,  6.16s/it]

In [ ]:
import numpy as np

train_lengths = [len(d["tokenized_text"]) for d in train_gold + train_silver + train_silver_2025]
dev_lengths = [len(d["tokenized_text"]) for d in json.load(open(PATH_DEV, 'r', encoding='utf-8'))]

def show_stats(name, lengths):
    arr = np.array(lengths)
    print(f"--- {name} ---")
    print(f"count:   {len(arr)}")
    print(f"min:     {arr.min()}")
    print(f"mean:    {arr.mean():.2f}")
    print(f"median:  {np.median(arr):.2f}")
    print(f"p75:     {np.percentile(arr, 75):.2f}")
    print(f"p90:     {np.percentile(arr, 90):.2f}")
    print(f"p95:     {np.percentile(arr, 95):.2f}")
    print(f"p99:     {np.percentile(arr, 99):.2f}")
    print(f"max:     {arr.max()}")
    print()

    for cutoff in [256, 384, 512, 640, 768, 1024]:
        over = (arr > cutoff).sum()
        pct = over / len(arr) * 100
        print(f">{cutoff}: {over}/{len(arr)} ({pct:.2f}%)")
    print()

show_stats("TRAIN", train_lengths)
show_stats("DEV", dev_lengths)

--- TRAIN ---
count:   1949
min:     53
mean:    313.50
median:  304.00
p75:     363.00
p90:     436.00
p95:     495.60
p99:     602.52
max:     2696

>256: 1369/1949 (70.24%)
>384: 386/1949 (19.81%)
>512: 76/1949 (3.90%)
>640: 15/1949 (0.77%)
>768: 4/1949 (0.21%)
>1024: 1/1949 (0.05%)

--- DEV ---
count:   80
min:     130
mean:    329.86
median:  316.00
p75:     376.00
p90:     459.00
p95:     512.15
p99:     655.03
max:     768

>256: 61/80 (76.25%)
>384: 19/80 (23.75%)
>512: 4/80 (5.00%)
>640: 1/80 (1.25%)
>768: 0/80 (0.00%)
>1024: 0/80 (0.00%)

